In [26]:
import yfinance as yf
import pandas as pd
import numpy as np
import time
from datetime import date
import os
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

In [27]:
NIFTY50_SAMPLE = [                                                      
    "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "INFY.NS", "ICICIBANK.NS",
    "HINDUNILVR.NS", "ITC.NS", "SBIN.NS", "BHARTIARTL.NS", "KOTAKBANK.NS",
    "LT.NS", "AXISBANK.NS", "ASIANPAINT.NS", "MARUTI.NS", "TITAN.NS",
]   
    # Each item is a ticker Symbol
    # NSE tickers need '.NS' suffix for yfinance
    # BSE would use '.BO' instead

NIFTY50_INDEX = "^NSEI"     # used as a market-wide feature

START_DATE = "2020-01-01"
END_DATE = date.today()
TEST_CUTOFF    = "2024-01-01"    # train = before, test = on/after

OUTPUT_DIR = "data/raw"     # folder where csv files will be saved

WINDOW = 60
HORIZON = 1
FETCH_PAUSE = 1.0

RAW_DIR        = "data/raw"
FEATURE_DIR    = "data/features"
SEQ_DIR        = "data/sequences"
INDEX_FILE     = os.path.join(RAW_DIR, "NSEI.csv")

FEATURE_COLS = [
    'log_return_1d', 'log_return_5d', 'log_return_10d', 'log_return_21d',
    'rsi_14', 
    'macd_line', 'macd_signal', 'macd_histogram',
    'close_to_sma20', 'close_to_sma50',
    'roc_5', 'roc_21',
    'bb_width', 'bb_position',
    'atr_14_norm', 'realised_vol_21',
    'volume_zscore', 'volume_ratio_5d', 'obv_zscore',
    'rolling_beta', 'index_ret', 'relative_return'
]
NORM_COLS      = [f"{c}_norm" for c in FEATURE_COLS]
TARGET_COL     = f"target_return_{HORIZON}d"

TRADING_DAYS  = 252
TX_COST       = 0.001        # 0.1% per trade (realistic NSE estimate)
RISK_FREE     = 0.065        # 6.5% annual (approx Indian 10-yr G-Sec yield)
RF_DAILY      = RISK_FREE / TRADING_DAYS


### STAGE I - DATA LOADING

NSE tickers need a ".NS" suffix for yfinance. BSE would use ".BO" instead.

In [28]:
def fetch_stock_data(ticker: str, start: str, end: str) -> pd.DataFrame:
    df = yf.download(
        ticker,
        start=start,                # start date
        end=end,                    # end date
        auto_adjust=True,           # fixes historical prices in case of stock splits
        progress=False,             # turn off loading bar
        multi_level_index=False     # we need simple columns
    )

    if df.empty:                    # sometimes yahoo returns nothing
        print(f"WARNING: no data returned for {ticker}")    # 
        return df

    df["Ticker"] = ticker           # add ticker column
    df.index.name = "Date"          # Pandas store date as index
    return df
    

def fetch_all(tickers: list[str], start: str, end: str, pause: float = 1.0) -> dict[ str, pd.DataFrame] :
    # It downloads stock data for multiple tickers, one after another, and stores each stock's data in a dictionary.
    """ 
        Pull data for a list of tickers with a small delay between calls
        to stay polite to Yahoo Finance's servers and avoid rate limiting
    """
    # tickers: list[str] => list of ticker symbols
    # start: str, end: str => start and end date, yfinance accepts start & end in str format
    # pause: float = 1.0 => to pause 1 sec after dowloading each stock data, yahoo finance blocks users with multiple requests

    data = {}   # dictionary to store each stock data

    for i, ticker in enumerate(tickers, 1):
        print(f"[{i}/{len(tickers)}] Fetching {ticker} ...")        # progress
        df = fetch_stock_data(ticker, start, end)                   # calls fetch stock data
        if not df.empty:
            data[ticker] = df
        time.sleep(pause)                                           # puase for 1 sec
    return data

def save_data(data: dict[str, pd.DataFrame], output_dir: str):      # save data in OUTPUT_DIR
    os.makedirs(output_dir, exist_ok = True)
    for ticker, df in data.items():
        clean_name = ticker.replace(".NS", "").replace("^", "")
        path = os.path.join(output_dir, f"{clean_name}.csv")
        df.to_csv(path)
        print(f" Saved{path} ({len(df)} rows)")


### Feature Engineering

In [29]:
RAW_DIR = "data/raw"
FEATURE_DIR = "data/feature"
INDEX_FILE = "data/raw/NSEI.csv"

1. Log Returns (4 features)

    You use log returns instead of percentage returns for two reasons:  
    they're additive across time periods (log_return_5d ≈ sum of 5 daily log returns), and they're closer to normally distributed —  
    both good properties for neural network inputs.

In [30]:
# Log Return Feature :
def add_return_features(df: pd.DataFrame) -> pd.DataFrame:
    """
        Log returns are preferred over simple percentage returns in quant finance
        because they're additive over time and more normally distributed —
        both useful properties for neural network inputs.
    
        log_return_1d  = log(Close_t / Close_{t-1})  ← most important feature
        log_return_5d  = log(Close_t / Close_{t-5})  ← weekly momentum
        log_return_10d = log(Close_t / Close_{t-10}) ← two-week momentum
        log_return_21d = log(Close_t / Close_{t-21}) ← monthly momentum
    """
    close = df['Close']

    df['log_return_1d'] = np.log(close/close.shift(1))
    df['log_return_5d'] = np.log(close/close.shift(5))
    df['log_return_10d'] = np.log(close/close.shift(10))
    df['log_return_21d'] = np.log(close/close.shift(21))

    return df

2. Momentum Indicators (7 features)

    RSI, MACD line, MACD signal, MACD histogram, price-to-SMA ratios, and rate-of-change.  
    The key decision is feeding all three MACD components separately rather than just using a  
    crossover signal — the LSTM can discover its own crossover logic from the raw components, which  
    is almost always more expressive.

     * **RSI**  
        RSI compares Average recent gains against Average recent losses  
        If gains dominate, RSI becomes large; If losses dominate, RSI becomes small.  

        Traditionally,  
            RSI > 70 => Overbought, stock has risen quickly  
            RSI < 30 => Oversold, stock has fallen rapidly  

        Notice these are not guarantees.

    * **MACD**
        Moving Average Convergence Divergence
        - measures the strength of price movements,  
        MACD measures the relationship between two moving averages.
        - But questions arise:  
            - Is the trend getting stronger?  
            - Is it slowing down?  
            - Is momentum increasing?  
            - Is the trend about to reverse?  
        MACD attempts to answer these questions.
        


In [31]:

# Momentum Indicators :
def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    """
        RSI (Relative Strength Index) — measures speed and magnitude of price moves.
        Range: 0–100. Above 70 = overbought, below 30 = oversold.
        We return the raw RSI value (not a signal) so the LSTM can learn
        its own thresholds from data rather than using our hardcoded ones.
    
        We implement manually here to avoid a TA-Lib dependency.
    """

    # input => series = close prises
    # output => RSI values
    # period => time interval = 14days, industry standard

    delta = series.diff()               # Computes current price - perivous
    
    gain = delta.clip(lower = 0)        # clip(lower = 0) => if less than zero => becomes zero      
    loss = delta.clip(upper = 0)        # clip(upper = 0) => this keeps only negative moments

    # RSI compares avg recent gain vs avg recent loss
    avg_gain = gain.ewm(        # this computes EMA of Gains
        alpha = 1 / period,     
        # This controls how quickly older observations lose influence
        # small alpha => smooth RSI, large alpha more sensitive RSI
        min_periods = period,       # first 13 rows become NAN, becuase there is no historical data
        adjust = False 
        ).mean()
    
    avg_loss = loss.ewm(        # this computes EMA for Loss
        alpha = 1 / period,
        min_periods = period,
        adjust = False
        ).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1+ rs))
    return rsi

def compute_macd(series: pd.Series,
                 fast: int = 12, slow: int = 26, signal: int = 9):
    """
        MACD (Moving Average Convergence Divergence).
        Returns three series:
        macd_line      = EMA(fast) - EMA(slow)
        macd_signal    = EMA(macd_line, signal)
        macd_histogram = macd_line - macd_signal (crossover signal)

        We feed all three to the LSTM separately so it can learn
        which component of MACD is predictive for each stock.
    """
    # Fast EMA (typically 12 days)
    ema_fast = series.ewm(span = fast, adjust = False).mean()

    # Slow EMA (typically 26 days)
    ema_slow = series.ewm(span = slow, adjust = False).mean()

    # MACD Line => difference between short term and long term trends
    macd_line = ema_fast - ema_slow

    # Signal Line is 9 day EMA of the MACD Line
    macd_sig = macd_line.ewm(span = signal, adjust = False).mean()

    """Histogram => measure of distance between MACD and Signal
    Interpretation:
    Positive Histogram: MACD is above the Signal Line → bullish momentum is strengthening.
    Negative Histogram: MACD is below the Signal Line → bearish momentum is strengthening.
    Histogram near zero: Momentum is weakening or changing direction."""
    macd_hist = macd_line - macd_sig

    return macd_line, macd_sig, macd_hist

def add_momentum_features(df: pd.DataFrame) -> pd.DataFrame:
    df['rsi_14'] = compute_rsi(df['Close'], period = 14)

    macd_line, macd_sig, macd_hist = compute_macd(df['Close'])
    df['macd_line'] = macd_line
    df['macd_signal'] = macd_sig
    df['macd_histogram'] = macd_hist

    # Price relative to its own moving averages — captures trend state
    df['close_to_sma20'] = df['Close']/df['Close'].rolling(20).mean() - 1
    df['close_to_sma50'] = df['Close']/df['Close'].rolling(50).mean() - 1

    # Rateof change - normalized momentum over N days
    df['roc_5'] = df['Close'].pct_change(5)
    df['roc_21'] = df['Close'].pct_change(21)

    return df



3. Volatility Indicators (4 features)

    Bollinger Band width (how much is the stock compressing before a breakout?), BB position (where is  
    price within the band?), ATR normalised by price (raw vol scaled to be comparable across stocks at  
    different price levels), and 21-day realised vol annualised. Volatility features matter because LSTM  
    return predictions are significantly more reliable during low-vol regimes than high-vol ones.

    * **Bollinger Bands** 
        - How far is today's price from its recent average?
        - Bollinger Bands :
          - Middle Band => SMA20
          - Upper Band => SMA20 + 2 * std20
          - Lower Band => SMA20 - 2 * std20

In [32]:
def add_volatility_features(df: pd.DataFrame) -> pd.DataFrame:
    """ 
        Bollinger Bands — price relative to its own volatility envelope.
        bb_width    = how wide the bands are (realized vol proxy)
        bb_position = where price sits within the band (-1 to +1 roughly)

        ATR (Average True Range) — raw volatility in price terms.
        Normalised by Close so it's comparable across different price levels.

        Realised vol — rolling std of log returns, annualised.
        A direct measure of how much the stock is moving day-to-day.
    """

    close = df["Close"]
    high = df["High"]
    low = df["Low"]

    # Bollinger Bands (20-dyas, 2 std)
    sma20 = close.rolling(20).mean()
    std20 = close.rolling(20).std()
    bb_upper = sma20 + 2 * std20
    bb_lower = sma20 - 2 * std20

    # Bollinger Band width => measures the width of volatility envolope
    df['bb_width'] = (bb_upper - bb_lower)/ sma20
    
    # Bollinger Position => Where the current price sits within the bands
    # Near 0 = close to the lower band, near 1 = close to the upper band.
    df['bb_position'] = (close - bb_lower)/(bb_upper - bb_lower)

    # ATR
    # ATR (Average True Range) is a technical indicator that measures how much a stock typically moves in a day,
    # regardless of whether it moves up or down.
    prev_close = close.shift(1)
    true_range = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs(),
    ], axis = 1).max(axis = 1)
    df['atr_14_norm'] = true_range.ewm(span = 14, adjust = False).mean() / close

    # Realised volatility (annualised)
    log_ret = np.log(close / close.shift(1))
    df['realised_vol_21'] = log_ret.rolling(21).std()*np.sqrt(252)

    return df


4. Volume Signals (3 features)

    Volume z-score (the core signal — how abnormal is today's volume?), volume ratio vs 5-day average  
    (simpler version), and OBV z-score (On-Balance Volume, a directional accumulation proxy). Volume is  
    your best cheap proxy for institutional activity without buying expensive order flow data.

In [33]:
def add_volume_features(df: pd.DataFrame) -> pd.DataFrame:
    """ 
        Volume z-score: how unusual is today's volume vs the past 20 days?
        Values above +2 or below -2 signal abnormal activity.
        This is a simple proxy for institutional flow / news events.

        OBV (On-Balance Volume): cumulates volume with sign based on price direction.
        Captures whether volume is accumulating (bullish) or distributing (bearish).
        We normalise OBV by its own rolling mean so it's stationary enough for LSTM.
    """

    volume = df['Volume']
    close = df['Close']

    # Volume z-score (rolling 20-day)
    vol_mean = volume.rolling(20).mean()
    vol_std = volume.rolling(20).std()
    df['volume_zscore'] = (volume - vol_mean) / vol_std.replace(0, np.nan)

    # Volume ratio — today vs 5-day average (simpler, more stable)
    df['volume_ratio_5d'] = volume/ volume.rolling(5).mean()

    # OBV (normalised)
    direction = np.sign(close.diff())
    obv = (volume * direction).cumsum()
    obv_mean = obv.rolling(20).mean()
    obv_std = obv.rolling(20).std()
    df['obv_zscore'] = (obv - obv_mean) / obv_std.replace(0, np.nan)

    return df




5. Market Context (3 features)

    This is the most quant-specific feature group. Rolling 60-day beta tells you how much the stock  
    co-moves with Nifty. The relative return (stock return minus beta-adjusted index return) is the  
    idiosyncratic component — what the market doesn't explain. This is what quant models are actually  
    trying to predict, not the raw return.

In [34]:
def add_market_features(df: pd.DataFrame, index_df: pd.DataFrame) -> pd.DataFrame:
    """ 
        Beta-adjusted relative return:
        = stock's log return - (rolling_beta × index log return)

        This tells you how much the stock moved BEYOND what the market explains.
        Positive = stock is outperforming the market today (idiosyncratic strength).
        Negative = stock is underperforming even accounting for market movement.

        Rolling beta (60-day) is recalculated every day using only past data —
        no lookahead bias.
    """
    index_ret = np.log(index_df['Close'] / index_df['Close'].shift(1))
    index_ret.name = 'index_ret'

    stock_ret = df['log_return_1d']

    # Align on date index
    aligned = pd.concat([stock_ret, index_ret], axis = 1).dropna()

    # Rolling 60-day beta = cov(stock, index) / var(index)
    rolling_cov = aligned['log_return_1d'].rolling(60).cov(aligned['index_ret'])
    rolling_var = aligned['index_ret'].rolling(60).var()
    rolling_beta = rolling_cov / rolling_var.replace(0, np.nan)

    df['rolling_beta'] = rolling_beta
    df['index_ret'] = index_ret
    df['relative_return'] = stock_ret - (rolling_beta * index_ret)

    return df

### Target Variable

In [35]:
def add_target(df : pd.DataFrame, horizon: int = 1) -> pd.DataFrame:
    """ 
    Target = log return N days ahead.
    We use shift(-horizon) to peek forward — this is intentional ONLY for the
    target column. All feature columns must never use future data.

    For classification (direction): target_direction = 1 if return > 0 else 0
    We keep both — you can switch loss functions to test regression vs clf.
    """

    df[f"target_return_{horizon}d"] = df[" log_return_1d"].shift(-horizon)
    df[f"target_direction_{horizon}"] = (df[f"target_return_{horizon}d"] > 0).astype(int)

    return df

### Rolling Z-Score Normalisation
avoids lookhead bias vs global min-max

In [36]:
def rolling_zscore_normalise(df : pd.DataFrame, cols : list[str], window : int = 252) -> pd.DataFrame:
    """ 
        Normalise each feature using its own rolling 252-day (1yr) mean and std.
        This is the correct approach for time-series ML:
        - Global normalisation uses future data to compute mean/std → LOOKAHEAD BIAS
        - Rolling normalisation only uses past data → SAFE

        Values more than 3 std from the mean are clipped to prevent extreme
        values destabilising LSTM training.
    """

    for col in cols:
        if col not in df.columns:
            continue
        roll_mean = df[col].rolling(window, min_periods = 60).mean()
        roll_std = df[col].rolling(window, min_periods = 60).mean()
        z = (df[col] - roll_mean) / roll_std.replace(0, np.nan)
        df[f"{col}_norm"] = z.clip(-3,3)
    return df

### STAGE II - ADD FEATURES

In [37]:
def build_features(ticker_csv : str,
                   index_df : pd.DataFrame,
                   horizon : int = 1) -> pd.DataFrame:
    df = pd.read_csv(ticker_csv, index_col = 'Date', parse_dates = True)

    df = add_return_features(df)
    df = add_momentum_features(df)
    df = add_volatility_features(df)
    df = add_volume_features(df)
    df = add_market_features(df, index_df)
    df = add_target(df, horizon = horizon)
    df = rolling_zscore_normalise(df, FEATURE_COLS)

    df = df.dropna(subset = [f"{c}_norm" for c in FEATURE_COLS if f"{c}_norm" in df.columns])

    return df


def add_features():

    os.makedirs(FEATURE_DIR, exist_ok = True)
    index_df = pd.read_csv(INDEX_FILE, index_col = "Date", parse_dates = True)
    raw_files = [f for f in os.listdir(RAW_DIR)
                 if f.endswith(".csv") and f != "NSEI.csv"]
    
    for fname in sorted(raw_files):
        ticker = fname.replace(".csv", "")
        df_feat = build_features(
            ticker_csv = os.path.join(RAW_DIR, fname),
            index_df = index_df,
            horizon = HORIZON,
        )
        out_path = os.path.join(FEATURE_DIR, f"{ticker}_features.csv")
        df_feat.to_csv(out_path)

    for i, col in enumerate(NORM_COLS, 1):
        print(f" {i:02d}.{col}")


### STAGE III - SEQUENCE WINDOWING


In [38]:
def make_sequences(feat_arr : np.ndarray,       # 2D numpy array containing input features
                   target_arr : np.ndarray,     # 1D numpy array containing target values
                   window : int                 # number of previous days the LSTM should use
                   ) -> tuple:
    """
        Sliding window across (T, F) feature matrix → LSTM cannot directly use this 2D matrix
        T  = Number of time steps (days)
        F = Number of features

        It requires (samples, window, features) which is 3D tensor

        Row i:
            X[i] = feat_arr[i : i + window]       this is slicing   
            shape (window, F)  ← 60-day history
            eg: windo = 60 and F = 12 => shape is (60 previous days, 12 features)

            y[i] = tgt_arr[i + window] is return on day no. (i + window + 1)
            scalar              ← next-day return

        NEVER shuffle these arrays — time ordering is the signal.
    """

    X, y = [], []

    for i in range(len(feat_arr) - window):
        X.append(feat_arr[i : i + window])
        y.append(target_arr[i + window])

    return np.array(X, dtype = np.float32), np.array(y, dtype = np.float32)



def date_split(df : pd.DataFrame,       # complete feature engineered DataFrame
               cutoff : str,            # Date separating train and test split
               feature_cols : list,     # Input features for LSTM
               target_col : str,        # column to predict
               window : int             # Number of historical days per sequence
               )-> tuple:  
    
    """
    It performs 3 main tasks :
        1. Splits the data into training and testing sets based on dates
        2. Converts each split into sliding window sequences by calling make_sequences()
        3. Returns the sequences along with their corresponding dates

    Hard date split — test is strictly after train.
    Sequences are built on each split independently to prevent
    boundary leakage (a sequence straddling the cutoff date).
    """

    train_df = df[df.index < cutoff].copy() # everything before cutoff is train
    test_df = df[df.index >= cutoff].copy() # everything after cutfoff is test

    print(f"    Train : {train_df.index.min().date()} → "
          f"{train_df.index.max().date()}  ({len(train_df)} days)")
    print(f"    Test  : {test_df.index.min().date()}  → "
          f"{test_df.index.max().date()}  ({len(test_df)} days)")
    
    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing feature columns: {missing}")
    
    X_train, y_train = make_sequences(
        train_df[feature_cols].values, train_df[target_col].values, window
    )

    X_test, y_test = make_sequences(
        test_df[feature_cols].values, test_df[target_col].values, window
    )
    """ 
    By calling make_sequences separately on train_df and test_df,
    the last training sequence ends at the last training day,
    and the first test sequence starts fresh from the first test day.
    No sequence ever straddles the boundary.
    """

    return (X_train, y_train, X_test, y_test,
            train_df.index[window:], test_df.index[window:])

def sanity_check(X_train, y_train, X_test, y_test, ticker):
    # Function is a validation function
    # does not modify the data
    print(f"    Shapes → X_train {X_train.shape} | X_test {X_test.shape}")

    # Check for NaNs in X_train
    assert not np.isnan(X_train).any(),"NaNs in X_train"

    # Check for NaNs in X_test
    assert not np.isnan(X_test).any(), "NaNs in X_test"

    # Check for NaNs in y_train
    assert not np.isnan(y_train).any(), "NaNs in y_train"

    # Check for NaNs in y_test
    assert not np.isnan(y_test).any(), "NaNs in y_test"

    # Verify Window size
    assert X_train.shape[1] == WINDOW, f"Window mismatch(expected {WINDOW})"

    # Verify number of features
    assert X_train.shape[2] == X_test.shape[2], "Feature count mismatch"

    print(f"ALL OKAY")

def save_sequences(ticker, X_train, y_train, X_test, y_test,
                   train_dates, test_dates):
    
    os.makedirs(SEQ_DIR, exist_ok = True)
    base = os.path.join(SEQ_DIR, ticker)

    np.save(f"{base}_X_train.npy", X_train)
    np.save(f"{base}_y_train.npy", y_train)
    np.save(f"{base}_X_test.npy", X_test)
    np.save(f"{base}_y_test.npy", y_test)

    pd.Series(train_dates, name = "date").to_csv(f"{base}_train_dates.csv", index = False)
    pd.Series(test_dates, name = "date").to_csv(f"{base}_test_dates.csv", index = False)

def load_sequences(ticker: str, seq_dir: str = SEQ_DIR):
    """
    Call this from Stage 4 (LSTM model):
        X_train, y_train, X_test, y_test, train_dates, test_dates = load_sequences("RELIANCE")
    """

    base = os.path.join(seq_dir, ticker)
    return (
        np.load(f"{base}_X_train.npy"),
        np.load(f"{base}_y_train.npy"),
        np.load(f"{base}_X_test.npy"),
        np.load(f"{base}_y_test.npy"),

        pd.read_csv(f"{base}_train_dates.csv", parse_dates = ["date"])["date"],
        pd.read_csv(f"{base}_test_dates.csv", parse_dates = ["dates"])["dates"]
    )

def run_stage3():

    feature_files = sorted([
        f for f in os.listdir(FEATURE_DIR) if f.endswith("_feature.csv")
    ])
    if not feature_files:
        raise FileNotFoundError(f"No features CSV in {FEATURE_DIR}. Run Stage 2 first")
    
    summary = []

    for fname in feature_files:
        ticker = fname.replace("_feature.csv", "")
        print(f"\n {ticker} ...")

        df = pd.read_csv(
            os.path.join(FEATURE_DIR, fname),
            index_col = "Date", parse_dates = True,
        ).dropna(subset = [TARGET_COL])

        X_train, y_train, X_test, y_test, train_dates, test_dates = date_split(
            df = df,
            cutoff = TEST_CUTOFF,
            feature_cols = NORM_COLS,
            target_col = TARGET_COL,
            window = WINDOW            
        )

        sanity_check(X_train, y_train, X_test, y_test, ticker)
        save_sequences(ticker, X_train, y_train, X_test, y_test, train_dates, test_dates)

        summary.append({
            "Ticker" : ticker,
            "Train seqs" : len(X_train),
            "Test seqs" : len(X_test),
            "Features" : X_train.shape[2]
        })
        print(f"    Saved to {SEQ_DIR}/{ticker}_*.npy")

        print("\n" + "=" * 60)
        print("Summary")
        print("=" * 60)
        print(pd.DataFrame(summary).to_string(index=False))

### STAGE IV - LSTM Model Architecture + Training

Loads sequences from stage 3 + Builds + Trains + Saves two models per ticker  
- regression : Predicts next-day log return (float)
- classifier : predicts direction (1 = up, 0 = down)

Both models share the same LSTM backbone - only the output head differs  
Train both and compare : sometimes direction accuracy matters more than  
minimising return error, depending on your trading strategy.

Output:  
    models/TICKER_regression.keras   
    models/TICKER_classifier.keras   
    models/TICKER_history.csv       ← loss curves for plotting

In [39]:
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import(
    LSTM, Dense, Dropout, BatchNormalization, Bidirectional
)

from tensorflow.keras.callbacks import(
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

import sys
# sys.path.append(os.path.dirname(os.path.abspath(__file__)))

MODEL_DIR = "models"
TICKERS     = [
    "RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK",
    "HINDUNILVR", "ITC", "SBIN", "BHARTIARTL", "KOTAKBANK",
    "LT", "AXISBANK", "ASIANPAINT", "MARUTI", "TITAN",
]

# Training Hyperparameters :
BATCH_SIZE = 32
MAX_EPOCHS = 100
LR = 1e-3
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

Regression LSTM — predicts next-day log return as a float.  

Architecture decisions explained:  

1. Bidirectional LSTM (first layer only):  
    Processes the sequence both forward AND backward, giving richer  
    context for each timestep. Only on the first layer because stacking  
     two bidirectional layers is expensive and rarely helps for daily data.  

2. return_sequences=True on layer 1, False on layer 2:  
    Layer 1 needs to pass the full sequence to layer 2 (return_sequences=True).  
    Layer 2 only passes its final hidden state to the Dense layers (False).  
    This is the standard stacked-LSTM pattern.  

3. Dropout (0.3) after each LSTM:  
    Prevents overfitting. Critical here because financial time series  
    are noisy and models overfit very easily.  

4. BatchNormalization before Dense layers:  
    Stabilises training when return magnitudes vary across stocks.  
    Keeps gradients from exploding or vanishing in the dense head.  

5. L2 regularisation on Dense layers (1e-4):  
    Additional overfitting control. Small value so it doesn't dominate loss.  

6. Linear output activation:  
    We're predicting a continuous return value, not a probability.  
    Never use sigmoid/relu on a regression output.  

Shape flow:  
- Input          : (batch, 60, 22)  
- iLSTM         : (batch, 60, 128)   ← 64 units × 2 directions  
- Dropout        : (batch, 60, 128)  
- LSTM           : (batch, 64)         ← final hidden state only  
- Dropout        : (batch, 64)  
- BatchNorm      : (batch, 64)  
- Dense(32, relu): (batch, 32)  
- Dense(1, linear): (batch, 1)        ← predicted return  _

MODEL :

In [40]:
# REGRESSION MODEL --------------------------------------------------------------------------------------
def build_regression_model(timesteps : int, n_features : int) -> Model:
    # timesteps =  window size(prevous trading days)
    # n_features = no. of columns
    # return type = Model

    inputs = Input(
        shape = (timesteps, n_features),    # tensorflow automatically handles batch size
        name = 'ohlcv_sequence'
    )

    # Layer 1 : Bidirectional LSTM
    """
    Why use bidirectional window?

    During training, the model already has the complete 60-day window.
    Reading it both forward and backward helps it learn richer relationships within that window.
    """
    x = Bidirectional(
        # creates two LSTMs 
                # one reads forward day 1-> day2 -> ... day 60
                # one reads backward day 60-> day59 -> ... day 1
        LSTM(
            # whatever learned from the the input sequence, gets converted to 64 values
            # Hidden state size = 64(comman starting number)
            64,
            return_sequences= True,
            # instead of returning only the output from last day, return everyday's output
            # why? => the 2nd LSTM layer needs the entire sequence
            dropout = 0.1,              # input dropout, 10% of input connections are randomly ignored
            recurrent_dropout = 0.1     # dropout 10% of memory connections        
            ),
            name = 'bilstm_1'
    )(inputs)
    
    # Layer 2 : Unidirectional LSTM (extracts final state)
    x = LSTM(
        64,
        return_sequences = False,
        dropout = 0.1,
        recurrent_dropout = 0.1,
        name = 'lstm_2'
    )(x)

    # Layer 3 : Dense Head
    x = BatchNormalization(name = 'batch_norm')(x)
    x = Dense(32, activation = 'relu', kernel_regularizer = l2(1e-4), name = 'dense_1')(x)
    x = Dropout(0.2, name = 'drop_3')(x)

    # Layer 4 : Output Layer
    outputs = Dense(1, activation = 'linear', name = 'return_output')(x)

    model = Model(inputs, outputs, name = 'LSTM_Regression')
    model.complie(
        optimizer = Adam(learning_rate = LR, clipnorm = 1.0),
        loss = 'huber',
        metrics = ["mae"]
    )

    return model

# CLASSIFICATION MODEL --------------------------------------------------------------------------------------
def build_classifier_model(timesteps : int, n_features : int) -> Model:
    """
        Classification LSTM — predicts direction (1=up, 0=down).

        Same backbone as regression but with a sigmoid output and binary
        cross-entropy loss. We add class_weight in training to handle the
        slight imbalance between up/down days in bull markets.

        Sigmoid output gives a probability — you can tune the threshold
        (default 0.5) to trade off precision vs recall in backtesting.
        E.g. only go long when probability > 0.6 for a more conservative signal.
    """
    input = Input(shape = (timesteps, n_features), name = 'ohlcv_sequenc')

    # Layer 1 : Bidirectional LSTM Layer
    x = Bidirectional(
        LSTM(64, return_sequences = True, dropout = 0.1, recurrent_dropout = 0.1),
        name = 'bilstm_1'
    )(input)
    x = Dropout(0.3, name = 'drop_1')(x)

    # Layer 2 : Unidirectional LSTM Layer
    x = LSTM(64, return_sequences = False, dropout = 0.1, recurrent_dropout = 0.1,
             name = 'lstm_2')(x)
    x = Dropout(0.3, name = 'drop_2')(x)

    # Layer 3 : Dense Head
    x = BatchNormalization(name = 'batch_norm')(x)
    x = Dense(32, activation = 'relu',
              kernel_regularizer = l2(1e-4), name = 'dense_1')(x)
    x = Dropout(0.2, name = 'drop_3')(x)

    # Layer 4 : Output Layer
    output = Dense(1, activation = 'sigmoid', name = 'direction_output')(x)

    model = Model(input, output, name = 'lstm_classifier')
    model.compile(
        optimizer = Adam(leraning_rate = LR, clipnorm = 1.0),
        loss = 'binary_crossentropy',
        metrics = ['accuracy']
    )

    return model

CALLBACKS :

In [41]:
def get_callbacks(model_path: str, monitor: str = "val_loss") -> list:
    """
    Three callbacks that prevent wasted training time and overfitting:

    EarlyStopping:
        Stops training when val_loss hasn't improved for 15 epochs.
        restore_best_weights=True means you get the best checkpoint,
        not the last (overfitted) one.

    ReduceLROnPlateau:
        Halves learning rate when val_loss plateaus for 7 epochs.
        Lets the model fine-tune after the big initial learning phase.
        min_lr=1e-6 prevents the LR from becoming uselessly tiny.

    ModelCheckpoint:
        Saves the best model to disk during training.
        If training crashes at epoch 80, you don't lose everything.
    """
    return[
        EarlyStopping(
            monitor = monitor,
            patience = 15,
            restore_best_weights = True,
            verbose = 1
        ),

        ReduceLROnPlateau(
            monitor = monitor,
            factor = 0.5,
            patience = 7,
            min_lr = 1e-6,
            verbose = 1
        ),

        ModelCheckpoint(
            filepath = model_path,
            monitor = monitor,
            save_best_only = True,
            verbose = 0
        )
    ]

Class Weight (For Classifier Only)

In [42]:
def compute_class_weights(y_train : np.ndarray) -> dict:
    """
    In a bull market, up days outnumber down days (~55/45).
    Without class weights, the classifier learns to predict 'up' always
    and achieves 55% accuracy trivially — not useful for trading.
    Weighting penalises misclassifying the minority class more heavily.
    """
    n_total = len(y_train)
    n_up = y_train.sum()
    n_down = n_total - n_up
    w_up = n_total / (2* n_up) if n_up > 0 else 1.0
    w_down = n_total / (2 * n_down) if n_down > 0 else 1.0
    
    # Why divide by 2? => Weight = Total Samples / (Number of classes x Samples in class)

    print(f"    Class weights → up: {w_up:.3f}  down: {w_down:.3f}  "
          f"(up days: {n_up/n_total:.1%})")
    return {1: w_up, 0: w_down}

Training :

In [43]:
def train_regression(X_train, y_train, X_test, y_test, ticker) -> dict:
    os.makedirs(MODEL_DIR, exist_ok = True)

    timesteps, n_features = X_train.shape[1], X_train.shape[2]
    model = build_regression_model(timesteps, n_features)       # Function with model architecture

    if ticker == TICKERS[0]:
        model.summary()

    model_path = os.path.join(MODEL_DIR, f"{ticker}_regression.keras")
    history = model.fit(
        X_train, y_train,
        validation_data = (X_test, y_test),
        epochs = MAX_EPOCHS,
        batch_size = BATCH_SIZE,
        callbacks = get_callbacks(model_path, monitor = 'val_loss'),
        verbose = 0,
        shuffle = False,    # NEVER shuffle time-series batches
    )

    print(f"    Stopped at epoch {len(history.history['loss'])}")
    print(f"    Best val_loss : {min(history.history['val_loss']):.6f}")
    print(f"    Best val_mae  : {min(history.history['val_mae']):.6f}")

    return history.history

def train_classifier(X_train, y_train_dir, X_test, y_test_dir, ticker) -> dict:
    timesteps, n_features = X_train.shape[1], X_train.shape[2]
    model = build_classifier_model(y_train_dir)

    model_path = os.path.join(MODEL_DIR, f"{ticker}_classifer.keras")
    class_weight = compute_class_weights(y_train_dir)

    history = model.fit(
        X_train, y_train_dir,
        validation_data = (X_test, y_test_dir),
        epochs = MAX_EPOCHS,
        batch_size = BATCH_SIZE,
        callbacks = get_callbacks(model_path, monitor = "val_accuracy"),
        class_weight = class_weight,
        verbose = 0,
        shuffle = False,    # NEVER shuffle time-series batches
    )

    print(f"    Stopped at epoch {len(history.history['loss'])}")
    print(f"    Best val_accuracy : {max(history.history['val_accuracy']):.4f}")

    return history.history

def save_history(req_hist: dict, clf_hist: dict, ticker: str):
    max_len = max(len(req_hist['loss']), len(clf_hist['loss']))

    def pad(lst):
        return lst + [np.nan] * (max_len - len(lst))
    
    df = pd.DataFrame({
        "reg_loss" : pad(reg_hist["loss"]),
        "reg_val_loss" : pad(reg_his["cal_loss"]),
        "reg_mae" : pad(reg_hist["mae"]),
        "clf_loss" : pad(clf_hist["loss"]),
        "clf_val_loss" : pad(clf_hist["val_loss"]),
        "clf_accuracy" : pad(clf_hist["accuracy"]),
        "clf_val_accuracy" : pad(clf_hist["val_accuracy"])
    })
    path = os.path.join(MODEL_DIR, f"{ticker}_history,csv")
    df.to_csv(path, index_label = "epoch")
    print(f"    History saved -> {path}")

Loading Trained Model :

In [44]:
def load_model(ticker: str, model_type: str = "regression") -> Model:
    """
    model_type: 'regression' or 'classifier'

    Usage in Stage 5:
        model = load_model("RELIANCE", "regression")
        y_pred = model.predict(X_test)
    """
    path = os.path.join(MODEL_DIR, f"{ticker}_{model_type}.keras")
    if not os.path.exists(path):
        raise FileNotFoundError(f"No save modelat {path}. Run Stage 4 first")
    return tf.keras.models.load_model(path)

### STAGE V - Quant Evaluation Suite

In [45]:
sys.path.append(os.getcwd())

RESULTS_DIR   = "results"
MODEL_DIR     = "models"

TRADING_DAYS = 252
TX_COST = 0.001         # tax
RISK_FREE = 0.065        
""" It is the return you can expect from an investment that is 
considered to have very little default risk.

In India, a common approximation is the yield 
on long-term Government Securities (G-Secs).
"""
RF_DAILY      = RISK_FREE / TRADING_DAYS

ML Metrics :

In [46]:
def ml_metrics(y_true : np.ndarray, y_pred : np.ndarray) -> dict:
    """ 
    R² < 0 means the model is worse than predicting the mean every day.
    For financial returns, R² of 0.01 - 0.05 is actually considered useful
    (markets are hard to predict — don't expect 0.9).
    """

    residuals = y_true - y_pred
    ss_res = (residuals ** 2).sum()
    ss_tot = ((y_true - y_true.mean()) ** 2).sum()

    return {
        "rsme" : float(np.sqrt(np.mean(residuals ** 2))),
        "mae"  : float(np.mean(np.abs(residuals))),
        "r2"   : float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan,
    }

Directional Accuracy : 

In [47]:
def directional_accuracy(y_true : np.ndarray, y_pred : np.ndarray) -> dict:
    """ 
    This function comapres actual returns and predicted returns
    and asks did the model correctly predict whether the market would go UP or DOWN
    
    It only cares about the direction 
    """

    true_dir = (y_true > 0).astype(int)     # eg : [0.9, 0.3, 0.4, -0.01, -0.05 ...] => [True, True, True, False, False ...]
    pred_dir = (y_pred > 0).astype(int)     # eg : [0.9, -0.3, 0.4, 0.01, -0.05 ...] => [True, False, True, True, False ...]
    correct = (true_dir == pred_dir)

    large_mask = np.abs(y_true) > 0.01
    large_acc = correct[large_mask].mean() if large_mask.sum() > 0 else np.nan

    up_mask = y_true > 0
    up_acc = correct[up_mask].mean() if up_mask.sum() > 0 else np.nan

    down_mask = y_true < 0
    down_acc = correct[down_mask].mean() if down_mask.sum() > 0 else np.nan

    return {
            "directional_accuracy"  : float(correct.mean()),
            "large_move_accuracy"   : float(large_acc),
            "up_day_accuracy"       : float(up_acc),
            "down_day_accuracy"     : float(down_acc),
            "n_large_moves"         : int(large_mask.sum()),
        }

In [48]:
def simulate_strategy(y_true : np.ndarray, y_pred : np.ndarray, dates : pd.DatetimeIndex, threshold : float = 0.0) -> pd.DataFrame:
    """ 
    This function awnsers to : 
    If I actually traded based on my model's predictions, how much money would I have made?
        Simulates a simple long/short strategy driven by the model's signal.
    
        Rules:
          If predicted return > +threshold  → long  (+1)
          If predicted return < -threshold  → short (-1)
          Otherwise                         → flat  (0)
    
        Transaction cost applied on every position change only.
        A 5-day long position pays tx cost once on entry and once on exit.
    
        strategy_return[t] = position[t] x actual_return[t] - tx_cost_if_traded
        """
    positions = []
    for pred in y_pred:
        if pred > threshold:
            positions.append(1)
        elif pred < -threshold:
            positions.append(-1)
        else:
            positions.append(0)

    position_changes = np.diff(positions, prepend = 0)
    # np.diff() => substracts consecutive values
    # prepend = 0? => the model correctly counts that as entering a trade.
    tx_costs = np.abs(position_changes) * TX_COST

    strategy_ret = positions * y_true - tx_costs

    buyhold_ret = y_true.copy()
    # y_true already contains the actual market return for each day.
    # so it says that if we dont sell then, y_true is equivalent to buyhold_ret

    df = pd.DataFrame({
            "date"          : dates,
            "actual_return" : y_true,
            "pred_return"   : y_pred,
            "position"      : positions,
            "strategy_ret"  : strategy_ret,
            "buyhold_ret"   : buyhold_ret,
        }).set_index("date")


    # If I started with ₹1 (or ₹100,000), how would my money grow over time?
    # Think of the number 1 as representing 100% of your existing money.
    df["strategy_equity"] = (1 + df["strategy_ret"]).cumprod()
    df["buyhold_equity"] = (1 + df["buyhold_ret"]).cumprod()
    # cumprod() => cummulative product

    return df   

Risk Return Matrix :

In [53]:
def sharpe_ratio(returns : np.ndarray, rf_daily : float  = RF_DAILY) -> float:
    """
        Sharpe = (mean_daily_excess_return / std_daily_return) x sqrt(252)
    
        Interpretation:
          < 0    : losing money after risk-free rate
          0 - 0.5  : weak
          0.5 - 1  : acceptable
          1 - 2    : good (most hedge funds target this)
          > 2    : excellent (check carefully for overfitting)
        """
    excess = returns  - rf_daily
    if excess.std() == 0:
        return 0.0
    return float((excess.mean() / excess.std()) * np.sqrt(TRADING_DAYS))

def sortino_ratio(returns : np.ndarray, rf_daily : float = RF_DAILY) -> float:
    """
        It calculates how much excess return a fund generates for every unit of downside risk taken. 
        A higher Sortino ratio indicates better, more efficient performance.
    """
    excess = returns - rf_daily
    downside = excess[excess < 0]

    if len(downside) == 0 or downside.std() == 0:
        return np.nan
    return float ((excess.mean() / downside.std()) * np.sqrt(TRADING_DAYS))


def max_drawdown(equity_curve : np.ndarray) -> float:
    # max_drawdown => The largest percentage drop from any previous peak to the next lowest point.
    rolling_max = np.maximum.accumulate(equity_curve)
    drawdowns = (equity_curve - rolling_max) / rolling_max

    return float(drawdowns.min())

def calmar_ratio(annualised_return : float, mdd: float) ->float:
    """
        Calmar = annualised_return / |max_drawdown|
        Measures return per unit of worst-case drawdown risk.
        > 1 means you're generating more return than your worst drawdown.
    """

    if mdd == 0:
        return np.nan
    return float(annualised_return / abs(mdd))

Information Coefficient

In [ ]:
def information_coefficient(y_true : np.ndarray, y_pred : np.ndarray) -> dict:
    """
    Information Coefficient = IC measures whether your model correctly ranks assets or returns.

    Inputs:
        y_true → actual future returns
        y_pred → model predictions

    Outputs :
        How predictive the model is
        Whether the prediction is statistically significant
        Whether the signal is consistent through time

    Why Spearman not Pearson:
        We care whether the model correctly RANKS days — predicts that
        a high-return day is higher than a low-return day.
        Rank correlation captures this. Linear correlation doesn't.
    
    Interpretation:
        IC = 0    : no predictive power
        IC = 0.05 : weak but usable (common in live quant funds)
        IC = 0.10 : strong
        IC > 0.15 : exceptional (check for lookahead bias)
    
    ICIR = mean(rolling IC) / std(rolling IC)
        High ICIR = consistent signal. More valuable than a high but
        volatile IC that averages well but is unreliable day-to-day.    
    """

    ic, p_value = stats.spearmanr(y_pred, y_true)

    n = len(y_true)
    rolling_ic = []

    for i in range(21, n):
        window_ic = stats.spearman(y_pred[i-21 : i], y_true[i-21 : i])
        rolling_ic.append(window_ic)

    rolling_ic = np.array(rolling_ic)
    icir = float(rolling_ic.mean() / rolling_ic.std()) if rolling_ic.std() > 0 else np.nan

    return {
        "ic"      : float(ic),
        "ic_pval" : float(p_value),
        "icir"    : float(icir),
    }

Quintile Analysis :

In [56]:
def quintile_analysis(y_true : np.ndarray, y_pred : np.ndarray) -> dict:
    """ 
    It awnsers to :
        "If I trusted my model and only bought the highest-ranked predictions,
        would I actually make more money than buying the lowest-ranked predictions?"
    
    A quintile simply means :
        Divide the ranked predictions into 5 equal groups.

    Rank predictions into 5 buckets. Check if top quintile outperforms bottom.
    
    This is how factor researchers validate signals at quant funds.
        A good signal shows a monotonic pattern:
        Q1 (lowest pred) → Q5 (highest pred) returns increase steadily.
    
    Spread = Q5_mean - Q1_mean
        > 0      : model distinguishes high from low return days
        > 0.001  : ~25% annualised spread — strong alpha
    """

    df = pd.DataFrame(
        {
            "pred" : y_pred,
            "actual" : y_true,
        }
    )
    df["quantile"] = pd.qcut(df["pred"], q = 5,
                             labels = ["Q1", "Q2", "Q3", "Q4", "Q5"])

    qmeans = df.groupby("quintile", observed = True)["actual"].mean()

    result = {f"quintile_{q}_mean_ret" : float(qmeans.get(q, np.nan))}

    q1 = qmeans.get("Q1", np.nan)
    q5 = qmeans.get("Q5", np.nan)
    result["quintile_spread"] = float(q5 - q1) if not (np.isnan(q1) or np.isnan(q5)) else np.nan
    result["monotonic"]       = bool(qmeans.is_monotonic_increasing)
    return result



Full Evaluation Of One Ticker :

In [ ]:
def evaluate_ticker(ticker : str) -> dict:
    print(f" Evaluating : {ticker}")

    X_train, y_train, X_test, y_test, train_dates, test_dates = load_sequences(ticker)

    model = load_model(ticker, "regression")
    y_pred_raw = model.predict(X_test, verbose = 0).flatten()
    y_true = y_test.flatten()


    print(f"Test samples : {len(y_true)}")
    print(f"Test period {test_dates.iloc[0].date()} to {test_dates.iloc[-1].date()} ")


    ml = ml_metrics(y_true, y_pred_raw)
    print(f"\n  ML metrics:")
    print(f"RMSE : {ml['rmse']:.6f}")
    print(f"MAE  : {ml['mae']:.6f}")
    print(f"R²   : {ml['r2']:.4f}  {'⚠ worse than mean' if ml['r2'] < 0 else ''}")


    da = directional_accuracy(y_true, y_pred_raw)
    print(f"\n  Directional accuracy:")
    print(f"Overall          : {da['directional_accuracy']:.2%}")
    print(f"Large moves only : {da['large_move_accuracy']:.2%}  (n={da['n_large_moves']})")
    print(f"Up days          : {da['up_day_accuracy']:.2%}")
    print(f"Down days        : {da['down_day_accuracy']:.2%}")


    ic_metrics = information_coefficient(y_true, y_pred_raw)
    print(f"\n  Information Coefficient:")
    print(f"IC   : {ic_metrics['ic']:.4f}  (p={ic_metrics['ic_pval']:.4f})")
    print(f"ICIR : {ic_metrics['icir']:.4f}")


    q_metrics = quintile_analysis(y_true, y_pred_raw)
    print(f"\n  Quintile analysis:")
    for q in ["Q1", "Q2", "Q3", "Q4", "Q5"]:
        print(f"    {q}: {q_metrics[f'quintile_{q}_mean_ret']:+.5f}")
    print(f"    Spread (Q5-Q1) : {q_metrics['quintile_spread']:+.5f}")
    print(f"    Monotonic      : {q_metrics['monotonic']}")


    equity_df    = simulate_strategy(y_true, y_pred_raw, test_dates)
    strat_rets   = equity_df["strategy_ret"].values
    buyhold_rets = equity_df["buyhold_ret"].values
    

### Main :